# IBGE Municipalities - Bronze Ingestion

## Imports

In [0]:
import requests
from datetime import datetime
from pyspark.sql.functions import col, current_timestamp, lit
import uuid

## Configuration

In [0]:
environment = "dev"

catalog = f"ecommerce_{environment}"
schema = "bronze"
table_name = "ibge_municipalities"
target_table = f"{catalog}.{schema}.{table_name}"
source_system = "ibge"
source_dataset = "municipalities"

run_id = str(uuid.uuid4())

api_url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"

## Call the IBGE API

In [0]:
response = requests.get(api_url, timeout=5)

if response.status_code != 200:
    raise RuntimeError(f"Expected 200, got {response.status_code}")

## Save response to volume

In [0]:
volume_path = f"//Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json"

with open(volume_path, "w", encoding="utf-8") as f:
    f.write(response.text)

In [0]:
spark_read_path = f"dbfs:{volume_path[1:]}"
source_df = spark.read.json(spark_read_path)

In [0]:
source_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- microrregiao: struct (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- mesorregiao: struct (nullable = true)
 |    |    |-- UF: struct (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- nome: string (nullable = true)
 |    |    |    |-- regiao: struct (nullable = true)
 |    |    |    |    |-- id: long (nullable = true)
 |    |    |    |    |-- nome: string (nullable = true)
 |    |    |    |    |-- sigla: string (nullable = true)
 |    |    |    |-- sigla: string (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- nome: string (nullable = true)
 |    |-- nome: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- regiao-imediata: struct (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- nome: string (nullable = true)
 |    |-- regiao-intermediaria: struct (nullable = true)
 |    |    |-- UF: struct (nullable = true)
 |    |    |    |-- id: long (n

In [0]:
source_df.count()

5571

In [0]:
bronze_df = source_df.select(
    col("id").alias("ibge_municipality_id"),
    col("nome").alias("municipality_name"),
    col("regiao-imediata.regiao-intermediaria.UF.sigla").alias("state_code"),
    col("_metadata.file_path").alias("source_file_path"),
    col("_metadata.file_modification_time").alias("source_file_modification_time")
)

## Add Bronze metadata

In [0]:
bronze_df = (
    bronze_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_run_id", lit(run_id))
    .withColumn("source_system", lit(source_system))
    .withColumn("source_dataset", lit(source_dataset))
)

## Write to Bronze

In [0]:
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

In [0]:
spark.table(target_table).printSchema()

root
 |-- ibge_municipality_id: long (nullable = true)
 |-- municipality_name: string (nullable = true)
 |-- state_code: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(spark.table(target_table).limit(5))

ibge_municipality_id,municipality_name,state_code,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
1100015,Alta Floresta D'Oeste,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities
1100023,Ariquemes,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities
1100031,Cabixi,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities
1100049,Cacoal,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities
1100056,Cerejeiras,RO,dbfs:/Volumes/ecommerce_dev/landing/raw_files/api/municipalities/municipalities_api_response_20260812_140452.json,2026-08-12T14:04:52.000Z,2026-08-12T14:04:59.716Z,f17b2c20-5550-4219-968a-94c88923a9ff,ibge,municipalities


In [0]:
spark.table(target_table).count()

5571